<a href="https://colab.research.google.com/github/24f3005028/MLP-ASSIGNMENTS/blob/main/MLP_W2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [20]:
import pandas as pd
import numpy as np
import gdown
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder, OneHotEncoder, StandardScaler

file_id = "1gZblGovF8W-2J4sC2WdPMpBNW3eRriRH"
url = f"https://drive.google.com/uc?id={file_id}"
gdown.download(url, "dataset.csv", quiet=True)

df = pd.read_csv("dataset.csv")
# Clean column names and standardize missing values
df.columns = df.columns.str.strip()
df.replace("Unknown", np.nan, inplace=True)
print(f"Correct dataset loaded and columns cleaned: {df.shape[0]} rows, {df.shape[1]} columns.")

Correct dataset loaded and columns cleaned: 10000 rows, 13 columns.


## Question 1
**Which of the following columns have object datatype?**

In [19]:
# Q1: Identifying object datatype columns in the correct gaming dataset
obj_cols = df.select_dtypes(include='object').columns.tolist()
print("Object Columns:", obj_cols)
display(df.dtypes)

Object Columns: ['Gender', 'Location', 'GameGenre', 'GameDifficulty', 'EngagementLevel']


,0
PlayerID,int64
Age,float64
Gender,object
Location,object
GameGenre,object
PlayTimeHours,float64
InGamePurchases,float64
GameDifficulty,object
SessionsPerWeek,int64
AvgSessionDurationMinutes,int64


## Question 2
**In this dataset, how many "Males" from "Europe" have made "InGamePurchases"?**

In [21]:
# Question 2
ans2 = df[(df['Gender'] == 'Male') & (df['Location'] == 'Europe') & (df['InGamePurchases'] == 1)].shape[0]
print(f"Q2: {ans2}")

Q2: 299


## Question 3
**In your dataset, how many players under the "Age" 18 have strictly greater than 10 "PlayTimeHours"?**

In [22]:
# Question 3
ans3 = df[(df['Age'] < 18) & (df['PlayTimeHours'] > 10)].shape[0]
print(f"Q3: {ans3}")

Q3: 453


## Question 4
**How many total null values were present in the whole dataset?**

In [6]:
total_nulls = df.isna().sum().sum()
print(f"Total null values: {total_nulls}")

Total null values: 5548


## Question 5
**Which category has the least value counts in `y_train`?**

In [23]:
# Question 5
X = df.drop(columns=['EngagementLevel'])
y = df['EngagementLevel']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

print("Q5 y_train counts:")
print(y_train.value_counts())

Q5 y_train counts:
EngagementLevel
Medium    3983
Low       2021
High      1996
Name: count, dtype: int64


## Question 6
**Write the sum of transformed (imputed) "Age" column of the test dataset.**

In [24]:
# Question 6
train = X_train.copy()
test = X_test.copy()

for col in ['Age', 'Location', 'GameDifficulty', 'InGamePurchases']:
    train[col] = train[col].replace('Unknown', np.nan)
    test[col] = test[col].replace('Unknown', np.nan)

train['Age'] = pd.to_numeric(train['Age'], errors='coerce')
test['Age'] = pd.to_numeric(test['Age'], errors='coerce')
age_mean = train['Age'].mean()
train['Age'] = train['Age'].fillna(age_mean)
test['Age'] = test['Age'].fillna(age_mean)

print(f"Q6 (Sum of test Age): {round(test['Age'].sum(), 2)}")

Q6 (Sum of test Age): 63585.24


## Question 7
**Apply preprocessing and calculate the sum of the first five rows of the transformed test matrix.**

In [25]:
# Question 7
train = X_train.copy()
test = X_test.copy()

for col in ['Age', 'Location', 'GameDifficulty', 'InGamePurchases']:
    train[col] = train[col].replace('Unknown', np.nan)
    test[col] = test[col].replace('Unknown', np.nan)

# Impute and clean
train['Age'] = pd.to_numeric(train['Age'], errors='coerce').fillna(train['Age'].mean())
test['Age'] = pd.to_numeric(test['Age'], errors='coerce').fillna(train['Age'].mean())
train['Location'] = train['Location'].fillna('Other')
test['Location'] = test['Location'].fillna('Other')
train['GameDifficulty'] = train['GameDifficulty'].fillna(train['GameDifficulty'].mode()[0])
test['GameDifficulty'] = test['GameDifficulty'].fillna(train['GameDifficulty'].mode()[0])
train['InGamePurchases'] = pd.to_numeric(train['InGamePurchases'], errors='coerce').fillna(0)
test['InGamePurchases'] = pd.to_numeric(test['InGamePurchases'], errors='coerce').fillna(0)

train = train.drop(columns=['PlayerID'])
test = test.drop(columns=['PlayerID'])

# Encode and Scale
oe = OrdinalEncoder(categories=[['Easy', 'Medium', 'Hard']])
train[['GameDifficulty']] = oe.fit_transform(train[['GameDifficulty']])
test[['GameDifficulty']] = oe.transform(test[['GameDifficulty']])

cat_cols = ['Gender', 'Location', 'GameGenre']
ohe = OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore')
train_cat = ohe.fit_transform(train[cat_cols])
test_cat = ohe.transform(test[cat_cols])

train_final = np.hstack([train.drop(columns=cat_cols).values, train_cat])
test_final = np.hstack([test.drop(columns=cat_cols).values, test_cat])

scaler = StandardScaler()
test_scaled = scaler.fit(train_final).transform(test_final)

print(f"Q7 (Sum of first 5 rows): {round(test_scaled[:5].sum(), 2)}")

Q7 (Sum of first 5 rows): -7.17
